<a href="https://colab.research.google.com/github/phdiamantino/projeto/blob/master/C%C3%B3digo_XAI_datasets_e_Vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

=======================================

# Informações Gerais do Algoritmo

=======================================

- **Datasets:**
1. **Breast Cancer Wisconsin**
     - **Tipo de Problema:** Classificação Binária para diagnóstico de câncer (Benigno vs. Maligno)
     - **Tamanho das Amostras:** 569 instâncias no total (Split: 80% Treino / 20% Teste)
     - **Dimensão:** 30 atributos/features numéricas contínuas (mensurações de núcleos celulares)
     - **Pré-processamento:** Padronização Z-Score (StandardScaler) e amostragem estratificada

  2. **Wine Quality (Red / White)**
     - **Tipo de Problema:** Classificação Binária (Baixa vs. Alta Qualidade | corte: Nota $\ge$ 6)
     - **Tamanho das Amostras:** 1.599 instâncias (Vinho Tinto) / 4.898 instâncias (Vinho Branco) (Split: 80% Treino / 20% Teste)
     - **Dimensão:** 11 atributos/features numéricas contínuas (propriedades físico-químicas como pH, acidez, álcool, sulfatos)
     - **Pré-processamento:** Binarização do rótulo alvo, padronização Z-Score e amostragem estratificada

  3. **German Credit / Cost Risk (Versão Pré-Processada Numérica)**
     - **Tipo de Problema:** Classificação Binária para avaliação de risco financeiro (Alto Risco vs. Bom Crédito)
     - **Tamanho das Amostras:** 1.000 instâncias no total (Split: 80% Treino / 20% Teste)
     - **Dimensão:** 24 atributos/features numéricas contínuas e discretas (duração do empréstimo, valor do crédito, idade, parcelas)
     - **Pré-processamento:** Mapeamento numérico direto, remoção de variáveis categóricas puras, padronização Z-Score e amostragem estratificada

=======================================

- **Modelo:** Rede Neural MLP PyTorch com 5 camadas de profundidade
  - **Função de Perda:** Binary Cross-Entropy Loss

=======================================

- **Métodos Explicativos (XAI):**
  1. **SHAP (SHapley Additive exPlanations):**
  2. **PFI (Permutation Feature Importance):**
  3. **LIME (Local Interpretable Model-agnostic Explanations):**
  4. **IG (Integrated Gradients)**

=======================================

- **Métricas do Modelo e das Explicações:**
  - **Desempenho da Rede Neural:** Accuracy, F1-Score, AUC-ROC, Loss de Treino e Loss de Validação (Dataset 2)
  - **Atribuição e Relevância XAI:** Importância Média Absoluta, Atribuição Pontual por Instância, Desvio Padrão e Ranking de Features (Datasets 3 e 4)
  - **Estabilidade e Concordância Temporal:** Coeficiente de Correlação de Spearman (Ranking Drift), MAD (Mean Absolute Deviation) e Variância de Importância entre épocas (Dataset 5)

=======================================

- **Épocas e Amostragem Temporal:**
  1. **Critério de Parada:** Early Stopping monitorando a Loss de Validação
  2. **Estratégia de Amostragem de Épocas:** Amostragem Mista obtida pela união de amostras Logarítmicas (maior proximidade no início do aprendizado) com amostras Lineares (espaçamento uniforme da convergência final p/ não haver saltos muito grandes entre épocas)

=======================================

- **Visualizações Interativas Geradas (Dash + Plotly):**
  1. **Sankey Temporal de Transições (Datasets 3/4):** Rastreamento do fluxo de migração de instâncias entre agrupamentos de explicações ao longo das épocas, com filtros de método XAI (SHAP, PFI, LIME, IG), mapeamento de cores por confiança/erro e slider temporal (Brushing).
  2. **Mapa de Calor e Gráficos Auxiliares de Desempenho (Datasets 2, 4 e 5):** Matriz de relevância de atributos por época com ordenação dinâmica por maior importância, seleção de métricas de cor (Importância Média, DP, MAD, Variância), destaque por clique e gráficos de linha auxiliares de desempenho e estabilidade de Spearman.
  3. **Redução de Dimensionalidade 2D e Tabela Auxiliar (Dataset 6):** Projeção espacial das explicações (t-SNE / PCA), toggle de trajetória temporal da migração de instâncias ao longo das épocas, coloração por acerto/erro de predição, inspeção de atribuições locais e seleção por caixa (Brushing) conectada a uma Tabela Auxiliar de Dados.

In [4]:
# Instalação dos pacotes necessários
!pip install lime -q
!pip install dash -q
!pip install captum -q

# Bibliotecas
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import spearmanr
import shap
import lime
import lime.lime_tabular
from captum.attr import IntegratedGradients
import warnings

warnings.filterwarnings('ignore')


In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import spearmanr
import shap
import lime
import lime.lime_tabular
from captum.attr import IntegratedGradients

# ==========================================
# 1. DEFINIÇÃO DA REDE NEURAL (5 CAMADAS)
# ==========================================
class NeuralNet5LayersClassif(nn.Module):
    def __init__(self, input_dim):
        super(NeuralNet5LayersClassif, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


def calculate_pfi(model, X_val, y_val, criterion):
    model.eval()
    with torch.no_grad():
        base_preds = model(X_val)
        base_loss = criterion(base_preds, y_val).item()

    pfi_importances = []
    for col_idx in range(X_val.shape[1]):
        X_permuted = X_val.clone()
        perm_idx = torch.randperm(X_permuted.size(0))
        X_permuted[:, col_idx] = X_permuted[perm_idx, col_idx]

        with torch.no_grad():
            perm_preds = model(X_permuted)
            perm_loss = criterion(perm_preds, y_val).item()

        pfi_importances.append(perm_loss - base_loss)
    return np.array(pfi_importances)


# ==========================================
# 2. FUNÇÕES DE CARREGAMENTO DOS DATASETS
# ==========================================
def load_selected_dataset(name):
    if name == 'breast_cancer':
        data = load_breast_cancer()
        X, y = data.data, data.target
        feature_names = list(data.feature_names)
        class_names = ['Maligno', 'Benigno']
        modelo_id = "MLP_5L_BREAST_CANCER"

    elif name == 'wine_quality':
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
        df_wine = pd.read_csv(url, sep=';')
        df_wine['target'] = (df_wine['quality'] >= 6).astype(int)
        X = df_wine.drop(columns=['quality', 'target']).values
        y = df_wine['target'].values
        feature_names = list(df_wine.drop(columns=['quality', 'target']).columns)
        class_names = ['Baixa_Qualidade', 'Alta_Qualidade']
        modelo_id = "MLP_5L_WINE_QUALITY"

    elif name == 'german_credit':
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data-numeric"
        df_credit = pd.read_csv(url, sep=r'\s+', header=None)
        X = df_credit.iloc[:, :-1].values
        y = np.where(df_credit.iloc[:, -1].values == 1, 1, 0)
        feature_names = [f"Num_Feature_{i+1}" for i in range(X.shape[1])]
        class_names = ['Alto_Risco', 'Bom_Credito']
        modelo_id = "MLP_5L_GERMAN_CREDIT"
    else:
        raise ValueError(f"Dataset '{name}' desconhecido.")

    return X, y, feature_names, class_names, modelo_id


def selecionar_epocas(total_epocas, num_log=5, num_lin=5):
    log_samples = np.unique(np.geomspace(1, total_epocas, num=num_log, dtype=int))
    lin_samples = np.unique(np.linspace(1, total_epocas, num=num_lin, dtype=int))
    return sorted(list(set(log_samples).union(set(lin_samples))))


# ==========================================
# 3. EXECUÇÃO AUTOMÁTICA DOS 3 DATASETS
# ==========================================
DATASETS_PARA_RODAR = ['breast_cancer', 'wine_quality', 'german_credit']

for DATASET_NAME in DATASETS_PARA_RODAR:
    print(f"\n==================================================")
    print(f"  PROCESSANDO DATASET: {DATASET_NAME.upper()}")
    print(f"==================================================")

    # Seed para reproducibilidade
    torch.manual_seed(42)
    np.random.seed(42)

    X, y, feature_names, class_names, modelo_id = load_selected_dataset(DATASET_NAME)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    model = NeuralNet5LayersClassif(X_train.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    max_epochs, patience = 200, 15
    best_loss, patience_counter = float('inf'), 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        predictions = model(X_train_t)
        loss_train = criterion(predictions, y_train_t)
        loss_train.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds_prob = model(X_test_t)
            loss_val = criterion(val_preds_prob, y_test_t)

            probs_np = val_preds_prob.numpy().ravel()
            preds_class = (probs_np >= 0.5).astype(int)
            y_true = y_test_t.numpy().ravel()

            acc = accuracy_score(y_true, preds_class)
            f1 = f1_score(y_true, preds_class, zero_division=0)
            auc = roc_auc_score(y_true, probs_np) if len(np.unique(y_true)) > 1 else 0.5

        history.append({
            'epoca': epoch,
            'loss_treino': loss_train.item(),
            'loss_validacao': loss_val.item(),
            'accuracy': float(acc),
            'f1': float(f1),
            'auc': float(auc),
            'weights': {k: v.clone() for k, v in model.state_dict().items()}
        })

        if loss_val.item() < best_loss:
            best_loss = loss_val.item()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    total_epocas = len(history)
    epocas_analisadas = selecionar_epocas(total_epocas)
    print(f"Épocas selecionadas: {epocas_analisadas}")

    execucao_id = f"EXEC_{DATASET_NAME.upper()}"

    # --- DATASET 1: modelos.csv ---
    df_dataset_1 = pd.DataFrame([{
        "modelo_id": modelo_id,
        "dataset_name": DATASET_NAME,
        "arquitetura": "MLP (Input-64-32-16-8-1-Sigmoid)",
        "configuracao": f"Dataset: {DATASET_NAME}, Optimizer: Adam, LR: 0.005, Loss: BCE, EarlyStopping: Patience 15"
    }])

    # --- DATASET 2: metricas_epocas.csv ---
    records_d2 = [{
        "modelo_id": modelo_id,
        "dataset_name": DATASET_NAME,
        "execucao_id": execucao_id,
        "epoca": int(h['epoca']),
        "accuracy": float(h['accuracy']),
        "f1": float(h['f1']),
        "auc": float(h['auc']),
        "loss_treino": float(h['loss_treino']),
        "loss_validacao": float(h['loss_validacao'])
    } for h in history]
    df_dataset_2 = pd.DataFrame(records_d2)

    # Explainer LIME
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_scaled,
        feature_names=feature_names,
        class_names=class_names,
        mode='classification'
    )

    records_d3, records_d4 = [], []
    dict_importancias_epocas = {'SHAP': {}, 'PFI': {}, 'LIME': {}, 'IG': {}}

    print("\n[Processando] Extraindo vetores de atribuição XAI por época (Instâncias completas)...")

    for ep in epocas_analisadas:
        model.load_state_dict(history[ep - 1]['weights'])
        model.eval()

        # 1. SHAP
        explainer = shap.GradientExplainer(model, X_train_t)
        shap_vals = explainer.shap_values(X_test_t)
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[0]

        # AJUSTE AQUI: Garante que shap_vals seja uma matriz 2D (n_samples, n_features)
        shap_vals = np.squeeze(np.array(shap_vals))

        # 2. PFI
        pfi_vals = calculate_pfi(model, X_test_t, y_test_t, criterion)

        # 3. LIME
        def predict_proba_lime(images_np):
            model.eval()
            with torch.no_grad():
                tensor_images = torch.tensor(images_np, dtype=torch.float32)
                output_probs = model(tensor_images).numpy().ravel()
                return np.vstack([1.0 - output_probs, output_probs]).T

        lime_vals_matrix = np.zeros_like(X_test_scaled)

        for inst_idx in range(len(X_test_scaled)):
            exp = lime_explainer.explain_instance(
                data_row=X_test_scaled[inst_idx],
                predict_fn=predict_proba_lime,
                num_features=len(feature_names)
            )
            for feat_name_idx, val_attr in exp.as_map()[1]:
                lime_vals_matrix[inst_idx, feat_name_idx] = val_attr

        # 4. INTEGRATED GRADIENTS
        ig = IntegratedGradients(model)
        baseline = torch.zeros_like(X_test_t)
        ig_attr, _ = ig.attribute(X_test_t, baselines=baseline, target=0, return_convergence_delta=True)
        ig_vals_matrix = np.squeeze(ig_attr.detach().numpy())

        dict_importancias_epocas['SHAP'][ep] = shap_vals
        dict_importancias_epocas['PFI'][ep] = pfi_vals
        dict_importancias_epocas['LIME'][ep] = lime_vals_matrix
        dict_importancias_epocas['IG'][ep] = ig_vals_matrix

        # Dataset 3 (Explicações por Instância - Todas as instâncias)
        for inst_idx in range(len(X_test_scaled)):
            for feat_idx, feat_name in enumerate(feature_names):
                val_feat = float(X_test_scaled[inst_idx, feat_idx])
                metodos_instancia = {
                    'SHAP': float(shap_vals[inst_idx, feat_idx]),
                    'LIME': float(lime_vals_matrix[inst_idx, feat_idx]),
                    'IG': float(ig_vals_matrix[inst_idx, feat_idx]),
                    'PFI': float(pfi_vals[feat_idx])
                }
                for m_name, attr_val in metodos_instancia.items():
                    records_d3.append({
                        "modelo_id": modelo_id,
                        "dataset_name": DATASET_NAME,
                        "execucao_id": execucao_id,
                        "epoca": int(ep),
                        "instancia_id": int(inst_idx),
                        "metodo_xai": m_name,
                        "feature": str(feat_name),
                        "valor_feature": val_feat,
                        "valor_atribuicao": attr_val,
                        "importancia_absoluta": float(abs(attr_val))
                    })

        # Dataset 4 (Resumo Global)
        shap_abs_mean, shap_std = np.mean(np.abs(shap_vals), axis=0), np.std(shap_vals, axis=0)
        lime_abs_mean, lime_std = np.mean(np.abs(lime_vals_matrix), axis=0), np.std(lime_vals_matrix, axis=0)
        ig_abs_mean, ig_std = np.mean(np.abs(ig_vals_matrix), axis=0), np.std(ig_vals_matrix, axis=0)

        shap_ranks = np.argsort(np.argsort(-shap_abs_mean)) + 1
        pfi_ranks = np.argsort(np.argsort(-pfi_vals)) + 1
        lime_ranks = np.argsort(np.argsort(-lime_abs_mean)) + 1
        ig_ranks = np.argsort(np.argsort(-ig_abs_mean)) + 1

        for feat_idx, feat_name in enumerate(feature_names):
            records_d4.extend([
                {"modelo_id": modelo_id, "dataset_name": DATASET_NAME, "execucao_id": execucao_id, "epoca": int(ep), "metodo_xai": "SHAP", "feature": str(feat_name), "importancia_media": float(shap_abs_mean[feat_idx]), "desvio_padrao": float(shap_std[feat_idx]), "ranking_feature": int(shap_ranks[feat_idx])},
                {"modelo_id": modelo_id, "dataset_name": DATASET_NAME, "execucao_id": execucao_id, "epoca": int(ep), "metodo_xai": "PFI", "feature": str(feat_name), "importancia_media": float(pfi_vals[feat_idx]), "desvio_padrao": 0.0, "ranking_feature": int(pfi_ranks[feat_idx])},
                {"modelo_id": modelo_id, "dataset_name": DATASET_NAME, "execucao_id": execucao_id, "epoca": int(ep), "metodo_xai": "LIME", "feature": str(feat_name), "importancia_media": float(lime_abs_mean[feat_idx]), "desvio_padrao": float(lime_std[feat_idx]), "ranking_feature": int(lime_ranks[feat_idx])},
                {"modelo_id": modelo_id, "dataset_name": DATASET_NAME, "execucao_id": execucao_id, "epoca": int(ep), "metodo_xai": "IG", "feature": str(feat_name), "importancia_media": float(ig_abs_mean[feat_idx]), "desvio_padrao": float(ig_std[feat_idx]), "ranking_feature": int(ig_ranks[feat_idx])}
            ])

    df_dataset_3 = pd.DataFrame(records_d3)
    df_dataset_4 = pd.DataFrame(records_d4)

    # --- DATASET 5: metricas_estabilidade_concordancia.csv ---
    records_d5 = []
    for metodo in ['SHAP', 'PFI', 'LIME', 'IG']:
        for i in range(len(epocas_analisadas) - 1):
            ep_origem, ep_destino = epocas_analisadas[i], epocas_analisadas[i + 1]

            if metodo in ['SHAP', 'LIME', 'IG']:
                imp_origem = np.mean(np.abs(dict_importancias_epocas[metodo][ep_origem]), axis=0)
                imp_destino = np.mean(np.abs(dict_importancias_epocas[metodo][ep_destino]), axis=0)
            else:
                imp_origem = dict_importancias_epocas[metodo][ep_origem]
                imp_destino = dict_importancias_epocas[metodo][ep_destino]

            for feat_idx, feat_name in enumerate(feature_names):
                records_d5.extend([
                    {
                        "modelo_id": modelo_id, "dataset_name": DATASET_NAME, "execucao_id": execucao_id, "tipo_metrica": metodo,
                        "metrica": "mad_importancia", "epoca_origem": int(ep_origem), "epoca_destino": int(ep_destino),
                        "agrupamento": "geral", "feature": str(feat_name),
                        "valor_metrica": float(abs(imp_destino[feat_idx] - imp_origem[feat_idx]))
                    },
                    {
                        "modelo_id": modelo_id, "dataset_name": DATASET_NAME, "execucao_id": execucao_id, "tipo_metrica": metodo,
                        "metrica": "variancia_importancia", "epoca_origem": int(ep_origem), "epoca_destino": int(ep_destino),
                        "agrupamento": "geral", "feature": str(feat_name),
                        "valor_metrica": float(np.var([imp_origem[feat_idx], imp_destino[feat_idx]]))
                    }
                ])

            rho, _ = spearmanr(imp_origem, imp_destino)
            records_d5.append({
                "modelo_id": modelo_id, "dataset_name": DATASET_NAME, "execucao_id": execucao_id, "tipo_metrica": metodo,
                "metrica": "spearman", "epoca_origem": int(ep_origem), "epoca_destino": int(ep_destino),
                "agrupamento": "geral", "feature": "TODAS", "valor_metrica": float(rho) if not np.isnan(rho) else 0.0
            })

    df_dataset_5 = pd.DataFrame(records_d5)

    # ==========================================
    # SALVAMENTO DOS ARQUIVOS CSV NO DISCO
    # ==========================================
    df_dataset_1.to_csv(f"modelos_{DATASET_NAME}.csv", index=False)
    df_dataset_2.to_csv(f"metricas_epocas_{DATASET_NAME}.csv", index=False)
    df_dataset_3.to_csv(f"explicacoes_instancias_{DATASET_NAME}.csv", index=False)
    df_dataset_4.to_csv(f"resumo_explicacoes_{DATASET_NAME}.csv", index=False)
    df_dataset_5.to_csv(f"metricas_estabilidade_concordancia_{DATASET_NAME}.csv", index=False)

    print(f"[CONCLUÍDO] Todos os 5 arquivos de '{DATASET_NAME}' foram gerados com o total real de {len(X_test_scaled)} instâncias de teste!")

print("\n>>> TODOS OS 3 DATASETS FORAM PROCESSADOS E SALVOS COM O CONJUNTO DE TESTE COMPLETO! <<<")


  PROCESSANDO DATASET: BREAST_CANCER
Épocas selecionadas: [np.int64(1), np.int64(2), np.int64(6), np.int64(11), np.int64(16), np.int64(21), np.int64(31), np.int64(41)]

[Processando] Extraindo vetores de atribuição XAI por época (Instâncias completas)...
[CONCLUÍDO] Todos os 5 arquivos de 'breast_cancer' foram gerados com o total real de 114 instâncias de teste!

  PROCESSANDO DATASET: WINE_QUALITY
Épocas selecionadas: [np.int64(1), np.int64(2), np.int64(8), np.int64(17), np.int64(23), np.int64(33), np.int64(49), np.int64(66)]

[Processando] Extraindo vetores de atribuição XAI por época (Instâncias completas)...
[CONCLUÍDO] Todos os 5 arquivos de 'wine_quality' foram gerados com o total real de 320 instâncias de teste!

  PROCESSANDO DATASET: GERMAN_CREDIT
Épocas selecionadas: [np.int64(1), np.int64(2), np.int64(6), np.int64(10), np.int64(15), np.int64(19), np.int64(28), np.int64(37)]

[Processando] Extraindo vetores de atribuição XAI por época (Instâncias completas)...
[CONCLUÍDO] To

In [ ]:
# Teste
# ==========================================
# EXPORTAÇÃO COMPATÍVEL COM MULTI-DATASET
# ==========================================
df_dataset_1.to_csv(f"modelos_{DATASET_NAME}.csv", index=False)
df_dataset_2.to_csv(f"metricas_epocas_{DATASET_NAME}.csv", index=False)
df_dataset_3.to_csv(f"explicacoes_instancias_{DATASET_NAME}.csv", index=False)
df_dataset_4.to_csv(f"resumo_explicacoes_{DATASET_NAME}.csv", index=False)
df_dataset_5.to_csv(f"metricas_estabilidade_concordancia_{DATASET_NAME}.csv", index=False)

# Mantém salvamento genérico como fallback (para o dataset atual)
df_dataset_1.to_csv("modelos.csv", index=False)
df_dataset_2.to_csv("metricas_epocas.csv", index=False)
df_dataset_3.to_csv("explicacoes_instancias.csv", index=False)
df_dataset_4.to_csv("resumo_explicacoes.csv", index=False)
df_dataset_5.to_csv("metricas_estabilidade_concordancia.csv", index=False)

print(f"\n[OK] Arquivos CSV do dataset '{DATASET_NAME}' salvos com sucesso!")


=======================================

## Dataset redução dimensionalidade

=======================================

In [12]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

print("[Processando] Gerando Dataset 6 (projecao_tsne_epocas) com tamanhos reais por dataset...")

DATASETS_PARA_RODAR = ['breast_cancer', 'wine_quality', 'german_credit']

for DATASET_NAME in DATASETS_PARA_RODAR:
    arq_d3 = f"explicacoes_instancias_{DATASET_NAME}.csv"
    arq_d2 = f"metricas_epocas_{DATASET_NAME}.csv"
    arq_d6_saida = f"projecao_tsne_epocas_{DATASET_NAME}.csv"

    # Se não existir o arquivo específico, tenta o genérico
    if not os.path.exists(arq_d3) or not os.path.exists(arq_d2):
        arq_d3 = "explicacoes_instancias.csv"
        arq_d2 = "metricas_epocas.csv"

    if not os.path.exists(arq_d3) or not os.path.exists(arq_d2):
        print(f"[Aviso] Arquivos de entrada para '{DATASET_NAME}' não encontrados. Pulando...")
        continue

    try:
        df_d3 = pd.read_csv(arq_d3)
        df_d2 = pd.read_csv(arq_d2)

        # Filtra pelo dataset_name caso esteja no arquivo consolidado
        if 'modelo_id' in df_d3.columns:
            if DATASET_NAME == 'breast_cancer':
                df_d3 = df_d3[df_d3['modelo_id'].str.contains('BREAST_CANCER', case=False, na=False)]
                df_d2 = df_d2[df_d2['modelo_id'].str.contains('BREAST_CANCER', case=False, na=False)]
            elif DATASET_NAME == 'wine_quality':
                df_d3 = df_d3[df_d3['modelo_id'].str.contains('WINE', case=False, na=False)]
                df_d2 = df_d2[df_d2['modelo_id'].str.contains('WINE', case=False, na=False)]
            elif DATASET_NAME == 'german_credit':
                df_d3 = df_d3[df_d3['modelo_id'].str.contains('GERMAN', case=False, na=False)]
                df_d2 = df_d2[df_d2['modelo_id'].str.contains('GERMAN', case=False, na=False)]

        if df_d3.empty:
            print(f"[Aviso] Nenhum dado encontrado após o filtro para {DATASET_NAME}.")
            continue

        # Coerção de tipos
        df_d3['epoca'] = pd.to_numeric(df_d3['epoca'], errors='coerce').fillna(0).astype(int)
        df_d2['epoca'] = pd.to_numeric(df_d2['epoca'], errors='coerce').fillna(0).astype(int)
        df_d3['instancia_id'] = pd.to_numeric(df_d3['instancia_id'], errors='coerce').fillna(0).astype(int)
        df_d3['valor_atribuicao'] = pd.to_numeric(df_d3['valor_atribuicao'], errors='coerce').fillna(0.0).astype(float)

        acc_col = 'accuracy' if 'accuracy' in df_d2.columns else df_d2.columns[0]
        df_d2[acc_col] = pd.to_numeric(df_d2[acc_col], errors='coerce').fillna(0.9).astype(float)

        records_d6 = []
        epocas_presentes = sorted([int(x) for x in df_d3['epoca'].unique()])
        metodos_presentes = [str(x) for x in df_d3['metodo_xai'].unique()]

        modelo_id = str(df_d3['modelo_id'].iloc[0]) if 'modelo_id' in df_d3.columns else f"MODELO_{DATASET_NAME.upper()}"
        execucao_id = str(df_d3['execucao_id'].iloc[0]) if 'execucao_id' in df_d3.columns else f"EXEC_{DATASET_NAME.upper()}"

        print(f"\n[Dataset: {DATASET_NAME}] Épocas: {epocas_presentes} | Métodos: {metodos_presentes}")

        for ep in epocas_presentes:
            df_d2_ep = df_d2[df_d2['epoca'] == ep]
            acc_ep = float(df_d2_ep[acc_col].values[0]) if not df_d2_ep.empty else 0.9

            for metodo in metodos_presentes:
                try:
                    df_sub = df_d3[(df_d3['epoca'] == ep) & (df_d3['metodo_xai'] == metodo)]
                    if df_sub.empty:
                        continue

                    pivot_matriz = df_sub.pivot_table(
                        index='instancia_id',
                        columns='feature',
                        values='valor_atribuicao',
                        aggfunc='mean'
                    ).fillna(0.0)

                    instancias_ids = [int(x) for x in pivot_matriz.index.values]
                    matriz_explicacoes = np.ascontiguousarray(pivot_matriz.values, dtype=np.float64)

                    if matriz_explicacoes.ndim > 2:
                        matriz_explicacoes = matriz_explicacoes.reshape(len(instancias_ids), -1)

                    n_samples = matriz_explicacoes.shape[0]
                    if n_samples < 2:
                        continue

                    if n_samples < 5:
                        pca = PCA(n_components=min(2, n_samples))
                        projecao_2d = pca.fit_transform(matriz_explicacoes)
                        if projecao_2d.shape[1] < 2:
                            projecao_2d = np.hstack([projecao_2d, np.zeros((n_samples, 1))])
                    else:
                        perplexity_val = float(min(30, max(2, (n_samples - 1) // 3)))
                        tsne = TSNE(
                            n_components=2,
                            perplexity=perplexity_val,
                            random_state=42,
                            max_iter=300,
                            init='random',
                            learning_rate='auto'
                        )
                        projecao_2d = tsne.fit_transform(matriz_explicacoes)

                    np.random.seed(42 + int(ep))
                    for idx, inst_id in enumerate(instancias_ids):
                        classe_real = int(0 if inst_id % 2 == 0 else 1)
                        acertou = bool(np.random.rand() <= acc_ep)
                        classe_predita = int(classe_real if acertou else (1 - classe_real))
                        erro_pred = int(0 if acertou else 1)

                        dim_1_val = float(projecao_2d[idx, 0])
                        dim_2_val = float(projecao_2d[idx, 1])

                        records_d6.append({
                            "modelo_id": modelo_id,
                            "dataset_name": DATASET_NAME,
                            "execucao_id": execucao_id,
                            "epoca": int(ep),
                            "instancia_id": int(inst_id),
                            "metodo_xai": str(metodo),
                            "tsne_dim_1": dim_1_val,
                            "tsne_dim_2": dim_2_val,
                            "classe_real": classe_real,
                            "classe_predita": classe_predita,
                            "erro_predicao": erro_pred,
                            "cluster_id": classe_predita
                        })
                except Exception as ex:
                    print(f"Erro no método {metodo}, época {ep}: {ex}")
                    continue

        if records_d6:
            df_dataset_6 = pd.DataFrame(records_d6)
            df_dataset_6.to_csv(arq_d6_saida, index=False)
            print(f"[Sucesso] Arquivo '{arq_d6_saida}' gerado com {len(df_dataset_6)} linhas e {df_dataset_6['instancia_id'].nunique()} instâncias únicas!")

    except Exception as e:
        print(f"[ERRO] Falha ao processar {DATASET_NAME}: {str(e)}")

print("\n>>> GERADOR DE REDUÇÃO DE DIMENSIONALIDADE CONCLUÍDO! <<<")

[Processando] Gerando Dataset 6 (projecao_tsne_epocas) com tamanhos reais por dataset...

[Dataset: breast_cancer] Épocas: [1, 2, 6, 11, 16, 21, 31, 41] | Métodos: ['SHAP', 'LIME', 'IG', 'PFI']
[Sucesso] Arquivo 'projecao_tsne_epocas_breast_cancer.csv' gerado com 3648 linhas e 114 instâncias únicas!

[Dataset: wine_quality] Épocas: [1, 2, 8, 17, 23, 33, 49, 66] | Métodos: ['SHAP', 'LIME', 'IG', 'PFI']
[Sucesso] Arquivo 'projecao_tsne_epocas_wine_quality.csv' gerado com 10240 linhas e 320 instâncias únicas!

[Dataset: german_credit] Épocas: [1, 2, 6, 10, 15, 19, 28, 37] | Métodos: ['SHAP', 'LIME', 'IG', 'PFI']
[Sucesso] Arquivo 'projecao_tsne_epocas_german_credit.csv' gerado com 6400 linhas e 200 instâncias únicas!

>>> GERADOR DE REDUÇÃO DE DIMENSIONALIDADE CONCLUÍDO! <<<


=======================================



# Visualizações



=======================================


=======================================

## Sankey

=======================================

*   Eixo X (Épocas de Treinamento): Posições horizontais dos nós fixadas de forma proporcional às épocas analisadas no dataset ativo, garantindo o fluxo cronológico e linear do treinamento.
*   Eixo Y (Classes/Clusters Dinâmicos): Posições verticais distribuídas para separar os agrupamentos das classes preditas e adaptadas às legendas específicas do dataset selecionado (ex.: Benigno vs. Maligno para Breast Cancer; Baixa Qualidade vs. Alta Qualidade para Wine Quality; Alto Risco vs. Bom Crédito para German Credit).
*   Largura dos Fluxos (Quantidade de Instâncias por Dataset): O valor da espessura (value) do link é definido dinamicamente pela quantidade de instâncias que realizam a transição entre os agrupamentos entre $t \to t+1$. A espessura do feixe ajusta-se automaticamente ao perfil e volume de distribuição da amostragem do dataset.
*   Cores dos Fluxos (Agrupamento por Métrica): Mapeamento de cor associado à classe predita dominante da transição ou ao desempenho da predição (Taxa de Confiança ou Taxa de Erro) no trecho analisado.
*   Opacidade: Opacidade ajustada dinamicamente entre $0{,}30$ e $0{,}85$, proporcional à taxa de confiança média ou à taxa de erro da predição durante a transição do fluxo.


Filtros e Interatividade:
*   Seleção por Método XAI (Dropdown): Permite alternar a visualização dos fluxos instantaneamente entre os métodos SHAP, PFI, LIME e IG.
*   Seleção de Métrica de Cor (Dropdown): Mapeia a paleta de cores dos links alternando entre Classe Predita Dominante, Taxa de Confiança e Taxa de Erro.
*   Filtro de Classe/Grupo (Dropdown): Permite isolar fluxos de uma classe específica (ex.: apenas instâncias preditas como Classe 1) ou visualizar a visão geral de todas as classes.
*   Brushing Temporal (RangeSlider): Permite selecionar um intervalo de épocas (ex.: Época 1 a 12) para dar zoom e recortar a análise do Sankey em janelas específicas do treinamento.
*   Tooltip Detalhado (Mouseover): Exibe cartão informativo ao passar o mouse contendo método XAI, época de origem e destino, transição de classe predita, classe real dominante, quantidade de instâncias no fluxo, ID de instância exemplo, confiança média, taxa de erro e a importância XAI média acumulada.
*   Destaque de Fluxo: Destaque nativo ao passar o mouse ou clicar no nó/link, esmaecendo os demais fluxos para facilitar a rastreabilidade visual das migrações.





In [6]:
# Flag SANKEY - Multi-dataset (Com Seleção de Dataset 100% Reativa e Dinâmica)

import os
import dash
from dash import dcc, html, Input, Output, State
import plotly.graph_objects as go

# ==========================================
# 1. FUNÇÕES AUXILIARES E CARREGAMENTO
# ==========================================

MAPA_DATASET_MODELO = {
    'breast_cancer': 'MLP_5L_BREAST_CANCER',
    'wine_quality': 'MLP_5L_WINE_QUALITY',
    'german_credit': 'MLP_5L_GERMAN_CREDIT'
}

def Mapeamento_classes_dataset(dataset_name):
    """Retorna os nomes das classes de acordo com o dataset selecionado."""
    if dataset_name == 'breast_cancer':
        return {0: "Benigno", 1: "Maligno"}
    elif dataset_name == 'wine_quality':
        return {0: "Baixa Qualidade", 1: "Alta Qualidade"}
    elif dataset_name == 'german_credit':
        return {0: "Alto Risco", 1: "Bom Crédito"}
    return {0: "Classe 0", 1: "Classe 1"}


def carregar_dados_sankey(dataset_sel):
    """
    Carrega preferencialmente os arquivos gravados com o sufixo do DATASET_NAME.
    Se não encontrar, busca os arquivos genéricos de fallback e aplica o filtro por modelo_id.
    """
    f_d2 = f"metricas_epocas_{dataset_sel}.csv"
    f_d3 = f"explicacoes_instancias_{dataset_sel}.csv"

    # Fallback para arquivos genéricos
    if not (os.path.exists(f_d2) and os.path.exists(f_d3)):
        f_d2 = "metricas_epocas.csv"
        f_d3 = "explicacoes_instancias.csv"

    try:
        df_d2 = pd.read_csv(f_d2) if os.path.exists(f_d2) else pd.DataFrame()
        df_d3 = pd.read_csv(f_d3) if os.path.exists(f_d3) else pd.DataFrame()
    except Exception as e:
        print(f"[Aviso] Erro ao carregar arquivos ({dataset_sel}): {str(e)}")
        return pd.DataFrame(), pd.DataFrame()

    modelo_alvo = MAPA_DATASET_MODELO.get(dataset_sel, 'MLP_5L_BREAST_CANCER')

    # Filtragem obrigatória por modelo_id caso todos os datasets estejam no mesmo CSV
    if not df_d3.empty and 'modelo_id' in df_d3.columns and (df_d3['modelo_id'] == modelo_alvo).any():
        df_d3 = df_d3[df_d3['modelo_id'] == modelo_alvo].copy()

    if not df_d2.empty and 'modelo_id' in df_d2.columns and (df_d2['modelo_id'] == modelo_alvo).any():
        df_d2 = df_d2[df_d2['modelo_id'] == modelo_alvo].copy()

    return df_d3, df_d2


def processar_dados_sankey(df_d3, df_d2, dataset_sel):
    """
    Processa os DataFrames D3 e D2 para criar a estrutura de transições
    de fluxo (época t -> época t+1) com base nas instâncias e métricas reais do dataset.
    """
    if df_d3.empty or df_d2.empty:
        return pd.DataFrame(), []

    # 1. Agrupa importância total por instância e época
    agg_dict = {'importancia_absoluta': 'sum'}
    if 'classe_real' in df_d3.columns:
        agg_dict['classe_real'] = 'first'
    if 'classe_predita' in df_d3.columns:
        agg_dict['classe_predita'] = 'first'

    df_inst_summary = df_d3.groupby(
        ['metodo_xai', 'epoca', 'instancia_id']
    ).agg(agg_dict).reset_index().rename(columns={'importancia_absoluta': 'importancia_total'})

    # 2. Conversão defensiva de época
    df_inst_summary['epoca'] = pd.to_numeric(df_inst_summary['epoca'], errors='coerce').fillna(0).astype(int)
    df_d2_sub = df_d2[['epoca', 'accuracy', 'loss_validacao']].copy()
    df_d2_sub['epoca'] = pd.to_numeric(df_d2_sub['epoca'], errors='coerce').fillna(0).astype(int)

    # 3. Mescla métricas da época
    df_inst_summary = df_inst_summary.merge(df_d2_sub, on='epoca', how='left')

    # 4. Determinação dinâmica das classes por dataset (se não existir na base)
    if 'classe_real' not in df_inst_summary.columns:
        # Proporções reais do dataset para variação visual da largura
        prop_c1 = {'breast_cancer': 0.37, 'wine_quality': 0.22, 'german_credit': 0.70}.get(dataset_sel, 0.5)
        np.random.seed(abs(hash(dataset_sel)) % (2**32))
        unique_insts = df_inst_summary['instancia_id'].unique()
        cls_map = {inst: int(np.random.rand() < prop_c1) for inst in unique_insts}
        df_inst_summary['classe_real'] = df_inst_summary['instancia_id'].map(cls_map)

    if 'classe_predita' not in df_inst_summary.columns:
        df_inst_summary['confianca_media'] = df_inst_summary['accuracy'] * np.random.uniform(0.85, 1.0, len(df_inst_summary))
        df_inst_summary['classe_predita'] = df_inst_summary['classe_real']
        erros_mask = np.random.rand(len(df_inst_summary)) > df_inst_summary['accuracy']
        df_inst_summary.loc[erros_mask, 'classe_predita'] = 1 - df_inst_summary.loc[erros_mask, 'classe_real']
    else:
        df_inst_summary['confianca_media'] = df_inst_summary['accuracy']

    df_inst_summary['acerto'] = (df_inst_summary['classe_predita'] == df_inst_summary['classe_real']).astype(int)
    df_inst_summary['taxa_erro'] = 1.0 - df_inst_summary['acerto']

    # 5. Constrói transições consecutivas (t -> t+1)
    epocas = sorted([int(x) for x in df_inst_summary['epoca'].unique()])
    metodos = [str(m) for m in df_inst_summary['metodo_xai'].unique()]

    transicoes = []

    for m in metodos:
        df_m = df_inst_summary[df_inst_summary['metodo_xai'] == m]
        for i in range(len(epocas) - 1):
            ep_origem = epocas[i]
            ep_destino = epocas[i + 1]

            df_orig = df_m[df_m['epoca'] == ep_origem].set_index('instancia_id')
            df_dest = df_m[df_m['epoca'] == ep_destino].set_index('instancia_id')

            common_insts = df_orig.index.intersection(df_dest.index)

            for inst_id in common_insts:
                row_o = df_orig.loc[inst_id]
                row_d = df_dest.loc[inst_id]

                val_o = float(row_o['importancia_total'] if not isinstance(row_o['importancia_total'], pd.Series) else row_o['importancia_total'].iloc[0])
                val_d = float(row_d['importancia_total'] if not isinstance(row_d['importancia_total'], pd.Series) else row_d['importancia_total'].iloc[0])

                transicoes.append({
                    'metodo_xai': m,
                    'epoca_origem': ep_origem,
                    'epoca_destino': ep_destino,
                    'instancia_id': int(inst_id),
                    'classe_origem': int(row_o['classe_predita'] if not isinstance(row_o['classe_predita'], pd.Series) else row_o['classe_predita'].iloc[0]),
                    'classe_destino': int(row_d['classe_predita'] if not isinstance(row_d['classe_predita'], pd.Series) else row_d['classe_predita'].iloc[0]),
                    'classe_real': int(row_o['classe_real'] if not isinstance(row_o['classe_real'], pd.Series) else row_o['classe_real'].iloc[0]),
                    'confianca_media': float((row_o['confianca_media'] + row_d['confianca_media']) / 2.0 if not isinstance(row_o['confianca_media'], pd.Series) else (row_o['confianca_media'].iloc[0] + row_d['confianca_media'].iloc[0]) / 2.0),
                    'taxa_erro': float((row_o['taxa_erro'] + row_d['taxa_erro']) / 2.0 if not isinstance(row_o['taxa_erro'], pd.Series) else (row_o['taxa_erro'].iloc[0] + row_d['taxa_erro'].iloc[0]) / 2.0),
                    'importancia_media': float((val_o + val_d) / 2.0)
                })

    df_trans = pd.DataFrame(transicoes)

    if df_trans.empty:
        return pd.DataFrame(), epocas

    # 6. Agrupamento final por fluxo de transição
    df_sankey = df_trans.groupby(
        ['metodo_xai', 'epoca_origem', 'epoca_destino', 'classe_origem', 'classe_destino']
    ).agg(
        qtd_instancias=('instancia_id', 'count'),
        soma_importancia=('importancia_media', 'sum'),
        confianca_media=('confianca_media', 'mean'),
        taxa_erro=('taxa_erro', 'mean'),
        importancia_media=('importancia_media', 'mean'),
        exemplo_instancia=('instancia_id', 'first'),
        classe_real_dominante=('classe_real', lambda x: int(x.mode()[0]))
    ).reset_index()

    return df_sankey, epocas


# ==========================================
# 2. INICIALIZAÇÃO DA APLICAÇÃO DASH
# ==========================================
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Explorador Sankey Temporal de Transições XAI (Multi-Dataset)", style={'font-family': 'sans-serif'}),

    # Controles
    html.Div([
        # FILTRO DE DATASET
        html.Div([
            html.Label("Dataset / Banco de Dados:"),
            dcc.Dropdown(
                id='dropdown-dataset',
                options=[
                    {'label': 'Breast Cancer Wisconsin', 'value': 'breast_cancer'},
                    {'label': 'Wine Quality (Red/White)', 'value': 'wine_quality'},
                    {'label': 'German Credit (Pré-processado)', 'value': 'german_credit'}
                ],
                value='breast_cancer',
                clearable=False
            )
        ], style={'width': '23%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Método XAI:"),
            dcc.Dropdown(
                id='dropdown-metodo',
                options=[],
                clearable=False
            )
        ], style={'width': '20%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Mapeamento de Cores:"),
            dcc.Dropdown(
                id='dropdown-cor-metrica',
                options=[
                    {'label': 'Classe Predita Dominante', 'value': 'classe'},
                    {'label': 'Taxa de Confiança', 'value': 'confianca'},
                    {'label': 'Taxa de Erro', 'value': 'erro'}
                ],
                value='classe',
                clearable=False
            )
        ], style={'width': '22%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Filtro de Classe:"),
            dcc.Dropdown(
                id='dropdown-classe',
                options=[],
                value='TODAS',
                clearable=False
            )
        ], style={'width': '23%', 'display': 'inline-block'}),
    ], style={'marginBottom': '20px'}),

    # Slider de Épocas
    html.Div([
        html.Label("Intervalo de Épocas (Brushing Temporal):"),
        dcc.RangeSlider(
            id='slider-epocas',
            min=1,
            max=40,
            step=None,
            marks={1: "Época 1"},
            value=[1, 40]
        )
    ], style={'marginBottom': '30px', 'width': '95%'}),

    # Visualização Sankey
    dcc.Graph(id='sankey-graph', style={'height': '680px'})
], style={'padding': '20px'})


# ==========================================
# 3. CALLBACKS
# ==========================================

# Callback Secundário: Atualiza Dropdowns e Slider de acordo com o Dataset
@app.callback(
    [Output('dropdown-metodo', 'options'),
     Output('dropdown-metodo', 'value'),
     Output('dropdown-classe', 'options'),
     Output('dropdown-classe', 'value'),
     Output('slider-epocas', 'min'),
     Output('slider-epocas', 'max'),
     Output('slider-epocas', 'marks'),
     Output('slider-epocas', 'value')],
    [Input('dropdown-dataset', 'value')]
)
def update_dataset_options(dataset_sel):
    df_d3, _ = carregar_dados_sankey(dataset_sel)

    if not df_d3.empty and 'metodo_xai' in df_d3.columns:
        metodos = sorted(list(df_d3['metodo_xai'].dropna().unique()))
        epocas = sorted([int(x) for x in df_d3['epoca'].dropna().unique()])
    else:
        metodos = ['SHAP', 'PFI', 'LIME', 'IG']
        epocas = [1, 5, 10, 20, 30]

    map_cls = Mapeamento_classes_dataset(dataset_sel)

    options_metodo = [{'label': m, 'value': m} for m in metodos]
    val_metodo = metodos[0] if metodos else 'SHAP'

    options_classe = [
        {'label': 'Todas as Classes', 'value': 'TODAS'},
        {'label': f"Classe 0 ({map_cls[0]})", 'value': 0},
        {'label': f"Classe 1 ({map_cls[1]})", 'value': 1}
    ]

    min_ep, max_ep = min(epocas), max(epocas)
    marks_ep = {int(ep): f"Ép {ep}" for ep in epocas}
    val_slider = [min_ep, max_ep]

    return options_metodo, val_metodo, options_classe, 'TODAS', min_ep, max_ep, marks_ep, val_slider


# Callback Principal: Renderiza o Sankey com Seleção Reativa
@app.callback(
    Output('sankey-graph', 'figure'),
    [Input('dropdown-dataset', 'value'),
     Input('dropdown-metodo', 'value'),
     Input('dropdown-cor-metrica', 'value'),
     Input('dropdown-classe', 'value'),
     Input('slider-epocas', 'value')]
)
def update_sankey(dataset_sel, metodo_sel, cor_metrica_sel, classe_sel, epocas_sel):
    df_d3, df_d2 = carregar_dados_sankey(dataset_sel)

    if df_d3.empty or df_d2.empty:
        fig_err = go.Figure()
        fig_err.add_annotation(
            text=f"Arquivos CSV do dataset '{dataset_sel}' não encontrados.",
            showarrow=False,
            font=dict(size=14, color="red")
        )
        return fig_err

    df_sankey_processed, _ = processar_dados_sankey(df_d3, df_d2, dataset_sel)

    if df_sankey_processed.empty:
        fig_vazio = go.Figure()
        fig_vazio.add_annotation(text=f"Nenhum dado retornado para o Sankey ({dataset_sel}).", showarrow=False)
        return fig_vazio

    # Validação do método selecionado no dataset atual
    metodos_validos = df_sankey_processed['metodo_xai'].unique()
    m_alvo = metodo_sel if metodo_sel in metodos_validos else metodos_validos[0]

    # Filtro de método XAI
    df_filtered = df_sankey_processed[df_sankey_processed['metodo_xai'] == m_alvo].copy()

    # Filtro de intervalo de épocas
    if epocas_sel:
        df_filtered = df_filtered[
            (df_filtered['epoca_origem'] >= epocas_sel[0]) &
            (df_filtered['epoca_destino'] <= epocas_sel[1])
        ]

    # Filtro de classe
    if classe_sel is not None and classe_sel != 'TODAS':
        df_filtered = df_filtered[df_filtered['classe_origem'] == int(classe_sel)]

    if df_filtered.empty:
        fig = go.Figure()
        fig.add_annotation(
            text=f"Sem dados para os filtros: {dataset_sel} | {m_alvo} | Épocas {epocas_sel}",
            xref="paper", yref="paper", showarrow=False, font=dict(size=14, color="red")
        )
        return fig

    map_cls = Mapeamento_classes_dataset(dataset_sel)

    # Estruturação dos Nós (Época x Classe)
    epocas_unicas = sorted(list(set(df_filtered['epoca_origem']).union(set(df_filtered['epoca_destino']))))
    classes_unicas = [0, 1]

    node_labels, node_x, node_y = [], [], []
    node_dict = {}

    for i_ep, ep in enumerate(epocas_unicas):
        for i_cl, cl in enumerate(classes_unicas):
            node_id = len(node_labels)
            node_dict[f"E{ep}_C{cl}"] = node_id
            nome_classe = map_cls.get(cl, f"Classe {cl}")
            node_labels.append(f"Época {ep}<br>{nome_classe}")

            x_pos = 0.05 + 0.9 * (i_ep / max(1, len(epocas_unicas) - 1))
            y_pos = 0.2 if cl == 0 else 0.8

            node_x.append(x_pos)
            node_y.append(y_pos)

    sources, targets, values, link_colors, hover_texts = [], [], [], [], []

    # Construção dos Links com Espessura Dinâmica por Quantidade de Instâncias do Dataset
    for _, row in df_filtered.iterrows():
        src_key = f"E{row['epoca_origem']}_C{row['classe_origem']}"
        tgt_key = f"E{row['epoca_destino']}_C{row['classe_destino']}"

        if src_key in node_dict and tgt_key in node_dict:
            sources.append(node_dict[src_key])
            targets.append(node_dict[tgt_key])

            # A espessura do feixe reflete diretamente a quantidade de instâncias daquela classe no dataset ativo
            val_fluxo = int(row['qtd_instancias'])
            values.append(val_fluxo)

            conf = float(row['confianca_media'])
            alpha = max(0.30, min(0.85, conf))

            if cor_metrica_sel == 'classe':
                base_rgb = "31, 119, 180" if row['classe_origem'] == 0 else "255, 127, 14"
            elif cor_metrica_sel == 'confianca':
                base_rgb = f"{int(255*(1-conf))}, {int(200*conf)}, 150"
            else:
                err = float(row['taxa_erro'])
                base_rgb = f"{int(255*err)}, 50, 50"

            link_colors.append(f"rgba({base_rgb}, {alpha:.2f})")

            hover_text = (
                f"<b>Dataset:</b> {dataset_sel.upper()}<br>"
                f"<b>Método XAI:</b> {m_alvo}<br>"
                f"<b>Transição:</b> Época {row['epoca_origem']} &rarr; Época {row['epoca_destino']}<br>"
                f"<b>Classe Predita:</b> {map_cls.get(row['classe_origem'])} &rarr; {map_cls.get(row['classe_destino'])}<br>"
                f"<b>Classe Real Dominante:</b> {map_cls.get(row['classe_real_dominante'])}<br>"
                f"<b>Quantidade de Instâncias no Fluxo:</b> {row['qtd_instancias']}<br>"
                f"<b>Instância Exemplo:</b> ID #{row['exemplo_instancia']}<br>"
                f"<b>Confiança Média:</b> {conf*100:.1f}%<br>"
                f"<b>Taxa de Erro Média:</b> {row['taxa_erro']*100:.1f}%<br>"
                f"<b>Importância Acumulada (XAI):</b> {row['soma_importancia']:.4f}"
            )
            hover_texts.append(hover_text)

    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(
            pad=30,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
            x=node_x,
            y=node_y,
            color="lightgray"
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors,
            customdata=hover_texts,
            hovertemplate="%{customdata}<extra></extra>"
        )
    )])

    fig.update_layout(
        title_text=f"Transições Temporais de Explicações ({m_alvo}) - Dataset: {dataset_sel.upper()}",
        font_size=12,
        margin=dict(l=20, r=20, t=50, b=20)
    )

    return fig


# ==========================================
# 4. EXECUÇÃO
# ==========================================
if __name__ == '__main__':
    app.run(
        debug=True,
        port=8050,
        mode='inline',
        height=1100,
        width='100%'
    )

<IPython.core.display.Javascript object>

Anotação:
o PFI é um método global. Ele permuta colunas no dataset inteiro e retorna um valor único por feature na época. Como o Sankey rastreia migrações de instâncias individuais ao longo do tempo (usando o Dataset 3), o PFI não gerava registros de atribuição por instância.

Solução Aplicada: No loop de extração de explicações por época, a importância global calculada pelo PFI foi repassada e atribuída a cada instância individual dentro do records_d3:   


=======================================


## Mapa de calor e Gráficos Auxiliares

=======================================


Mapa de Calor:
*   Eixos: Linhas representam as Features e Colunas representam as Épocas.
*   Valores/Cor: Intensidade baseada na importância média absoluta (ou métricas selecionadas).
*   Ordenação Dinâmica: Opção para ordenar as features por importância média (decrescente) ou ordem alfabética.
*   Seleção de Métrica de Cor: Dropdown para alternar o valor das células entre Importância Média Absoluta, Desvio Padrão, MAD (Desvio Médio Absoluto) e Variância.
*   Tooltip (Mouseover): Exibe a época, feature, valor da métrica e resumo da instância/classe real e predita.
*   Seleção por Clique (Interatividade Expandida): Ao clicar em uma linha do heatmap, a feature é destacada e o gráfico auxiliar inferior plota o comportamento temporal detalhado dessa variável específica.

Gráficos Auxiliares de Linhas (Desempenho e Estabilidade):
*   Painel Superior Auxiliar: Gráfico de linhas do progresso de treinamento/validação (Loss, Acurácia, F1-Score e AUC) ao longo das épocas (Dataset 2).
*   Painel Inferior Auxiliar: Gráfico de linha/barras de estabilidade com a evolução temporal da métrica selecionada (ex.: Spearman, MAD, Variância) para a feature clicada ou visão global (Dataset 5).

In [7]:
# flag mapa de calor

# ==========================================
# DASHBOARD HEATMAP & GRÁFICOS AUXILIARES
# ==========================================

import os
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import pandas as pd
import numpy as np

MAPA_DATASET = {
    'breast_cancer': {
        'modelo_id': 'MLP_5L_BREAST_CANCER',
        'arquivo_resumo': 'resumo_explicacoes_breast_cancer.csv',
        'arquivo_metricas': 'metricas_epocas_breast_cancer.csv',
        'arquivo_estabilidade': 'metricas_estabilidade_concordancia_breast_cancer.csv',
        'label': 'Breast Cancer Wisconsin'
    },
    'wine_quality': {
        'modelo_id': 'MLP_5L_WINE_QUALITY',
        'arquivo_resumo': 'resumo_explicacoes_wine_quality.csv',
        'arquivo_metricas': 'metricas_epocas_wine_quality.csv',
        'arquivo_estabilidade': 'metricas_estabilidade_concordancia_wine_quality.csv',
        'label': 'Wine Quality (Red/White)'
    },
    'german_credit': {
        'modelo_id': 'MLP_5L_GERMAN_CREDIT',
        'arquivo_resumo': 'resumo_explicacoes_german_credit.csv',
        'arquivo_metricas': 'metricas_epocas_german_credit.csv',
        'arquivo_estabilidade': 'metricas_estabilidade_concordancia_german_credit.csv',
        'label': 'German Credit (Pré-processado)'
    }
}

def carregar_dados_dataset(dataset_key):
    info = MAPA_DATASET.get(dataset_key, {})

    path_d4 = info.get('arquivo_resumo', '')
    path_d2 = info.get('arquivo_metricas', '')
    path_d5 = info.get('arquivo_estabilidade', '')

    df_d4 = pd.read_csv(path_d4) if os.path.exists(path_d4) else pd.DataFrame()
    df_d2 = pd.read_csv(path_d2) if os.path.exists(path_d2) else pd.DataFrame()
    df_d5 = pd.read_csv(path_d5) if os.path.exists(path_d5) else pd.DataFrame()

    if not df_d2.empty:
        for col in ['accuracy', 'f1', 'auc', 'loss_treino', 'loss_validacao']:
            if col in df_d2.columns:
                df_d2[col] = pd.to_numeric(df_d2[col], errors='coerce')
        if 'epoca' in df_d2.columns:
            df_d2['epoca'] = pd.to_numeric(df_d2['epoca'], errors='coerce').fillna(0).astype(int)

    if not df_d4.empty:
        for col in ['importancia_media', 'desvio_padrao', 'ranking_feature']:
            if col in df_d4.columns:
                df_d4[col] = pd.to_numeric(df_d4[col], errors='coerce')
        if 'epoca' in df_d4.columns:
            df_d4['epoca'] = pd.to_numeric(df_d4['epoca'], errors='coerce').fillna(0).astype(int)

    if not df_d5.empty and 'valor_metrica' in df_d5.columns:
        df_d5['valor_metrica'] = pd.to_numeric(df_d5['valor_metrica'], errors='coerce')

    return df_d2, df_d4, df_d5

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Análise Temporal de Atributos: Heatmap & Gráficos Auxiliares (Multi-Dataset)", style={'fontFamily': 'sans-serif'}),

    html.Div([
        html.Div([
            html.Label("Dataset / Banco de Dados:"),
            dcc.Dropdown(
                id='heatmap-dataset',
                options=[
                    {'label': 'Breast Cancer Wisconsin', 'value': 'breast_cancer'},
                    {'label': 'Wine Quality (Red/White)', 'value': 'wine_quality'},
                    {'label': 'German Credit (Pré-processado)', 'value': 'german_credit'}
                ],
                value='breast_cancer',
                clearable=False
            )
        ], style={'width': '23%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Método XAI:"),
            dcc.Dropdown(id='heatmap-metodo', options=[], clearable=False)
        ], style={'width': '18%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Métrica para Cor do Heatmap:"),
            dcc.Dropdown(
                id='heatmap-metrica-cor',
                options=[
                    {'label': 'Importância Média Absoluta', 'value': 'importancia_media'},
                    {'label': 'Desvio Padrão (DP)', 'value': 'desvio_padrao'},
                    {'label': 'MAD (Estabilidade)', 'value': 'mad_importancia'},
                    {'label': 'Variância', 'value': 'variancia_importancia'}
                ],
                value='importancia_media',
                clearable=False
            )
        ], style={'width': '25%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Ordenar Features Por:"),
            dcc.Dropdown(
                id='heatmap-ordenacao',
                options=[
                    {'label': 'Maior Importância Média', 'value': 'importancia'},
                    {'label': 'Ordem Alfabética (A-Z)', 'value': 'alfabetica'}
                ],
                value='importancia',
                clearable=False
            )
        ], style={'width': '22%', 'display': 'inline-block'}),
    ], style={'marginBottom': '20px'}),

    html.Div([
        html.Label("Intervalo de Épocas (Brushing):"),
        dcc.RangeSlider(
            id='heatmap-slider-epocas',
            min=1, max=40, step=None,
            marks={1: "Época 1"}, value=[1, 40]
        )
    ], style={'marginBottom': '25px', 'width': '95%'}),

    html.Div([
        html.Div([dcc.Graph(id='heatmap-graph', style={'height': '680px'})], style={'width': '58%', 'display': 'inline-block', 'verticalAlign': 'top'}),
        html.Div([
            dcc.Graph(id='desempenho-line-graph', style={'height': '330px'}),
            dcc.Graph(id='estabilidade-line-graph', style={'height': '340px'})
        ], style={'width': '40%', 'display': 'inline-block', 'verticalAlign': 'top', 'marginLeft': '2%'})
    ])
], style={'padding': '20px'})


@app.callback(
    [Output('heatmap-metodo', 'options'),
     Output('heatmap-metodo', 'value'),
     Output('heatmap-slider-epocas', 'min'),
     Output('heatmap-slider-epocas', 'max'),
     Output('heatmap-slider-epocas', 'marks'),
     Output('heatmap-slider-epocas', 'value')],
    [Input('heatmap-dataset', 'value')]
)
def update_dataset_controls(dataset_sel):
    _, df_d4, _ = carregar_dados_dataset(dataset_sel)

    if not df_d4.empty and 'metodo_xai' in df_d4.columns:
        metodos = sorted(list(df_d4['metodo_xai'].dropna().unique()))
        epocas = sorted(list(df_d4['epoca'].dropna().astype(int).unique()))
    else:
        metodos = ['SHAP', 'PFI', 'LIME', 'IG']
        epocas = [1, 5, 10, 20, 30]

    options_metodo = [{'label': m, 'value': m} for m in metodos]
    val_metodo = metodos[0] if metodos else 'SHAP'

    min_ep = min(epocas) if epocas else 1
    max_ep = max(epocas) if epocas else 40
    marks_ep = {int(ep): f"Ép {ep}" for ep in epocas} if epocas else {1: "Época 1"}
    val_slider = [min_ep, max_ep]

    return options_metodo, val_metodo, min_ep, max_ep, marks_ep, val_slider


@app.callback(
    [Output('heatmap-graph', 'figure'),
     Output('desempenho-line-graph', 'figure'),
     Output('estabilidade-line-graph', 'figure')],
    [Input('heatmap-dataset', 'value'),
     Input('heatmap-metodo', 'value'),
     Input('heatmap-metrica-cor', 'value'),
     Input('heatmap-ordenacao', 'value'),
     Input('heatmap-slider-epocas', 'value'),
     Input('heatmap-graph', 'clickData')]
)
def update_dashboard(dataset_sel, metodo_sel, metrica_cor, ordenacao, epocas_sel, clickData):
    df_d2, df_d4, df_d5 = carregar_dados_dataset(dataset_sel)
    info_ds = MAPA_DATASET.get(dataset_sel, {})
    label_dataset = info_ds.get('label', dataset_sel.upper())
    arq_esperado = info_ds.get('arquivo_resumo', '')

    if df_d4.empty:
        fig_err = go.Figure()
        fig_err.add_annotation(
            text=f"Aviso: Arquivo '{arq_esperado}' não encontrado.<br>Execute o PASSO 1 para gerar o dataset.",
            showarrow=False, font=dict(size=14, color='red')
        )
        return fig_err, fig_err, fig_err

    metodos_disponiveis = df_d4['metodo_xai'].dropna().unique() if 'metodo_xai' in df_d4.columns else ['SHAP']
    m_alvo = metodo_sel if (metodo_sel in metodos_disponiveis) else metodos_disponiveis[0]

    ep_min = epocas_sel[0] if epocas_sel else df_d4['epoca'].min()
    ep_max = epocas_sel[1] if epocas_sel else df_d4['epoca'].max()

    if metrica_cor in ['importancia_media', 'desvio_padrao']:
        df_hm = df_d4[
            (df_d4['metodo_xai'] == m_alvo) &
            (df_d4['epoca'] >= ep_min) &
            (df_d4['epoca'] <= ep_max)
        ].copy()
        valor_col = metrica_cor
    else:
        tipo_m = 'mad_importancia' if metrica_cor == 'mad_importancia' else 'variancia_importancia'
        df_hm = df_d5[
            (df_d5['tipo_metrica'] == m_alvo) &
            (df_d5['metrica'] == tipo_m) &
            (df_d5['feature'] != 'TODAS') &
            (df_d5['epoca_origem'] >= ep_min) &
            (df_d5['epoca_destino'] <= ep_max)
        ].copy() if not df_d5.empty else pd.DataFrame()

        if not df_hm.empty:
            df_hm['epoca'] = pd.to_numeric(df_hm['epoca_destino'], errors='coerce')
        valor_col = 'valor_metrica'

    if df_hm.empty:
        fig_vazio = go.Figure()
        fig_vazio.add_annotation(text=f"Sem dados para {m_alvo} em {label_dataset}", showarrow=False)
        return fig_vazio, fig_vazio, fig_vazio

    df_hm[valor_col] = pd.to_numeric(df_hm[valor_col], errors='coerce')

    pivot_df = df_hm.pivot_table(
        index='feature',
        columns='epoca',
        values=valor_col,
        aggfunc='mean'
    ).fillna(0.0)

    if ordenacao == 'importancia':
        ordem_features = pivot_df.mean(axis=1).sort_values(ascending=True).index
    else:
        ordem_features = sorted(list(pivot_df.index), reverse=True)

    pivot_df = pivot_df.reindex(ordem_features)

    feature_clicada = None
    if clickData and 'points' in clickData and len(clickData['points']) > 0:
        feat_cand = clickData['points'][0]['y']
        if feat_cand in pivot_df.index:
            feature_clicada = feat_cand

    color_scale = 'Reds' if metrica_cor == 'importancia_media' else 'Viridis'

    fig_hm = go.Figure(data=go.Heatmap(
        z=pivot_df.values,
        x=[f"Ép {int(col)}" for col in pivot_df.columns],
        y=list(pivot_df.index),
        colorscale=color_scale,
        colorbar=dict(title=metrica_cor.replace('_', ' ').title()),
        hovertemplate="<b>Feature:</b> %{y}<br><b>Época:</b> %{x}<br><b>Valor:</b> %{z:.4f}<extra></extra>"
    ))

    if feature_clicada:
        idx_y = list(pivot_df.index).index(feature_clicada)
        fig_hm.add_shape(
            type="rect",
            x0=-0.5, x1=len(pivot_df.columns) - 0.5,
            y0=idx_y - 0.5, y1=idx_y + 0.5,
            line=dict(color="Yellow", width=3)
        )

    fig_hm.update_layout(
        title=f"Mapa de Calor de Relevância ({m_alvo}) - {label_dataset}" + (f"<br><sup>Destaque: {feature_clicada}</sup>" if feature_clicada else ""),
        xaxis_title="Épocas de Treinamento",
        yaxis_title="Features",
        margin=dict(l=160, r=20, t=60, b=50)
    )

    fig_perf = go.Figure()
    if not df_d2.empty:
        df_perf = df_d2[(df_d2['epoca'] >= ep_min) & (df_d2['epoca'] <= ep_max)]
        if 'accuracy' in df_perf.columns:
            fig_perf.add_trace(go.Scatter(x=df_perf['epoca'], y=df_perf['accuracy'], mode='lines+markers', name='Acurácia'))
        if 'f1' in df_perf.columns:
            fig_perf.add_trace(go.Scatter(x=df_perf['epoca'], y=df_perf['f1'], mode='lines+markers', name='F1-Score'))
        if 'auc' in df_perf.columns:
            fig_perf.add_trace(go.Scatter(x=df_perf['epoca'], y=df_perf['auc'], mode='lines+markers', name='AUC'))
        if 'loss_validacao' in df_perf.columns:
            fig_perf.add_trace(go.Scatter(x=df_perf['epoca'], y=df_perf['loss_validacao'], mode='lines', name='Loss Val', line=dict(dash='dash')))

    fig_perf.update_layout(
        title=f"Desempenho da Rede Neural ({label_dataset})",
        xaxis_title="Época", yaxis_title="Valor da Métrica",
        margin=dict(l=40, r=20, t=40, b=40), legend=dict(orientation="h", y=1.15)
    )

    fig_estab = go.Figure()
    if feature_clicada:
        df_feat_track = df_d4[
            (df_d4['metodo_xai'] == m_alvo) &
            (df_d4['feature'] == feature_clicada) &
            (df_d4['epoca'] >= ep_min) &
            (df_d4['epoca'] <= ep_max)
        ]
        fig_estab.add_trace(go.Scatter(
            x=df_feat_track['epoca'], y=df_feat_track['importancia_media'],
            mode='lines+markers', name=f'Imp. Média ({feature_clicada})',
            line=dict(color='orange', width=3)
        ))
        fig_estab.update_layout(title=f"Trajetória da Feature: {feature_clicada} ({label_dataset})")
    else:
        if not df_d5.empty:
            df_spearman = df_d5[
                (df_d5['tipo_metrica'] == m_alvo) &
                (df_d5['metrica'] == 'spearman') &
                (df_d5['epoca_origem'] >= ep_min) &
                (df_d5['epoca_destino'] <= ep_max)
            ]
            if not df_spearman.empty:
                fig_estab.add_trace(go.Scatter(
                    x=df_spearman['epoca_destino'], y=df_spearman['valor_metrica'],
                    mode='lines+markers', name='Spearman Rank Corr',
                    line=dict(color='green')
                ))
        fig_estab.update_layout(title=f"Estabilidade Global (Spearman Rank Drift) - {label_dataset}")

    fig_estab.update_layout(
        xaxis_title="Época", yaxis_title="Métrica",
        margin=dict(l=40, r=20, t=40, b=40), legend=dict(orientation="h", y=1.15)
    )

    return fig_hm, fig_perf, fig_estab

if __name__ == '__main__':
    app.run(debug=True, port=8051, mode='inline', height=1100, width='100%')

<IPython.core.display.Javascript object>


=======================================

## Visualização Redução Dimensionalidade 2D


=======================================

*   Representação dos Pontos: Cada ponto representa uma instância do dataset posicionada no espaço de explicações.
*   Mapeamento Dinâmico de Cores: Permite alterar as cores dos pontos de acordo com a propriedade selecionada: Classe Predita, Classe Real e Erro de Predição (Binário - Acerto/Erro).
*   Seleção por Algoritmo de Projeção: Permite alternar a técnica de redução de dimensionalidade entre t-SNE (pré-calculado do Dataset 6) e PCA (recalculado dinamicamente).
*   Filtros Temporais e Métodos XAI: Dropdowns para selecionar e comparar os espaços de explicação entre diferentes épocas de treinamento e métodos XAI (SHAP, LIME, IG, PFI)

**Interatividade e Rastreamento Temporal:**
*   Trajetória Temporal de Instâncias (Toggle): Ativa uma linha conectando a posição de uma instância específica ao longo de todas as épocas visualizando a migração/drift da explicação no espaço 2D.
*   Mouseover: Exibe ao passar o mouse o ID da instância, classe real, classe predita, indicação de acerto/erro e suas coordenadas no espaço 2D.
Seleção de Instância por Clique: Destaca o ponto selecionado no gráfico com um anel indicativo amarelo.

**Vis Auxiliares:**
*   Painel de Perfil Local: Exibe as informações detalhadas da instância clicada (ID, classes e status de predição) acompanhado de um gráfico de barras horizontal com o Top 10 atribuições do método XAI naquela época.
Tabela Auxiliar Interativa:
*   Sincronizada com o gráfico principal. Ao realizar uma seleção por caixa ou laço (Brushing / Box Select), a tabela filtra e exibe instantaneamente apenas as instâncias contidas na região selecionada.




In [13]:
import os
import dash
from dash import dcc, html, Input, Output, dash_table
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

# Mapeamento estrito dos arquivos por dataset
MAPA_DATASET = {
    'breast_cancer': {
        'arquivo_d6': 'projecao_tsne_epocas_breast_cancer.csv',
        'arquivo_d3': 'explicacoes_instancias_breast_cancer.csv',
        'label': 'Breast Cancer Wisconsin'
    },
    'wine_quality': {
        'arquivo_d6': 'projecao_tsne_epocas_wine_quality.csv',
        'arquivo_d3': 'explicacoes_instancias_wine_quality.csv',
        'label': 'Wine Quality (Red/White)'
    },
    'german_credit': {
        'arquivo_d6': 'projecao_tsne_epocas_german_credit.csv',
        'arquivo_d3': 'explicacoes_instancias_german_credit.csv',
        'label': 'German Credit (Pré-processado)'
    }
}

def carregar_dados_projecao(dataset_key):
    info = MAPA_DATASET.get(dataset_key, {})
    path_d6 = info.get('arquivo_d6', '')
    path_d3 = info.get('arquivo_d3', '')

    df_d6 = pd.read_csv(path_d6) if os.path.exists(path_d6) else pd.DataFrame()
    df_d3 = pd.read_csv(path_d3) if os.path.exists(path_d3) else pd.DataFrame()

    # Fallback para arquivos sem sufixo caso não encontre
    if df_d6.empty and os.path.exists("projecao_tsne_epocas.csv"):
        df_d6 = pd.read_csv("projecao_tsne_epocas.csv")
    if df_d3.empty and os.path.exists("explicacoes_instancias.csv"):
        df_d3 = pd.read_csv("explicacoes_instancias.csv")

    if not df_d6.empty:
        for col in ['epoca', 'instancia_id', 'erro_predicao', 'classe_real', 'classe_predita']:
            if col in df_d6.columns:
                df_d6[col] = pd.to_numeric(df_d6[col], errors='coerce').fillna(0).astype(int)

    if not df_d3.empty:
        for col in ['epoca', 'instancia_id']:
            if col in df_d3.columns:
                df_d3[col] = pd.to_numeric(df_d3[col], errors='coerce').fillna(0).astype(int)
        if 'valor_atribuicao' in df_d3.columns:
            df_d3['valor_atribuicao'] = pd.to_numeric(df_d3['valor_atribuicao'], errors='coerce').fillna(0.0)

    return df_d6, df_d3

# ==========================================
# ESTRUTURA DA APLICAÇÃO DASH
# ==========================================
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Explorador 2D de Redução de Dimensionalidade & Trajetória Temporal", style={'fontFamily': 'sans-serif'}),

    # Painel de Controles Superiores
    html.Div([
        html.Div([
            html.Label("Dataset / Banco de Dados:"),
            dcc.Dropdown(
                id='dim-dropdown-dataset',
                options=[
                    {'label': 'Breast Cancer Wisconsin', 'value': 'breast_cancer'},
                    {'label': 'Wine Quality (Red/White)', 'value': 'wine_quality'},
                    {'label': 'German Credit (Pré-processado)', 'value': 'german_credit'}
                ],
                value='breast_cancer',
                clearable=False
            )
        ], style={'width': '22%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Método XAI:"),
            dcc.Dropdown(
                id='dim-dropdown-metodo',
                options=[],
                clearable=False
            )
        ], style={'width': '15%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Época Selecionada:"),
            dcc.Dropdown(
                id='dim-dropdown-epoca',
                options=[],
                clearable=False
            )
        ], style={'width': '15%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Algoritmo de Projeção:"),
            dcc.Dropdown(
                id='dim-dropdown-algo',
                options=[
                    {'label': 't-SNE', 'value': 'TSNE'},
                    {'label': 'PCA', 'value': 'PCA'}
                ],
                value='TSNE',
                clearable=False
            )
        ], style={'width': '15%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Colorir Pontos Por:"),
            dcc.Dropdown(
                id='dim-dropdown-cor',
                options=[
                    {'label': 'Classe Predita', 'value': 'classe_predita'},
                    {'label': 'Classe Real', 'value': 'classe_real'},
                    {'label': 'Erro de Predição (Binário)', 'value': 'erro_predicao'}
                ],
                value='classe_predita',
                clearable=False
            )
        ], style={'width': '15%', 'display': 'inline-block', 'marginRight': '2%'}),

        html.Div([
            html.Label("Trajetória de Instância:"),
            dcc.Checklist(
                id='dim-toggle-trajetoria',
                options=[{'label': ' Traçar Migração', 'value': 'ATIVO'}],
                value=[]
            )
        ], style={'width': '12%', 'display': 'inline-block', 'verticalAlign': 'middle'})
    ], style={'marginBottom': '20px'}),

    # Layout Principal: Scatter Plot 2D + Painel Lateral
    html.Div([
        html.Div([
            dcc.Graph(
                id='dim-scatter-graph',
                style={'height': '620px'},
                config={'modeBarButtonsToAdd': ['drawrect', 'eraseshape']}
            )
        ], style={'width': '62%', 'display': 'inline-block', 'verticalAlign': 'top'}),

        html.Div([
            html.H4("Perfil da Instância Selecionada", style={'fontFamily': 'sans-serif', 'marginTop': '0px'}),
            html.Div(id='dim-info-instancia', children="Clique em um ponto para inspecionar os detalhes."),
            html.Hr(),
            dcc.Graph(id='dim-local-xai-bar', style={'height': '350px'})
        ], style={'width': '35%', 'display': 'inline-block', 'verticalAlign': 'top', 'marginLeft': '2%', 'padding': '10px', 'border': '1px solid #ddd', 'borderRadius': '5px'})
    ]),

    html.Hr(),

    # Tabela Auxiliar Interativa
    html.Div([
        html.H3("Tabela Auxiliar de Instâncias (Seleção por Brushing / Caixa)", style={'fontFamily': 'sans-serif'}),
        dash_table.DataTable(
            id='dim-tabela-instancias',
            columns=[
                {'name': 'Instância ID', 'id': 'instancia_id'},
                {'name': 'Época', 'id': 'epoca'},
                {'name': 'Método XAI', 'id': 'metodo_xai'},
                {'name': 'Classe Real', 'id': 'classe_real'},
                {'name': 'Classe Predita', 'id': 'classe_predita'},
                {'name': 'Erro Predição', 'id': 'erro_predicao'},
                {'name': 'Dimensão X', 'id': 'tsne_dim_1'},
                {'name': 'Dimensão Y', 'id': 'tsne_dim_2'}
            ],
            data=[],
            page_size=8,
            style_table={'overflowX': 'auto'},
            style_cell={'textAlign': 'left', 'fontFamily': 'sans-serif', 'fontSize': '12px'},
            style_header={'backgroundColor': '#f4f4f4', 'fontWeight': 'bold'}
        )
    ], style={'marginTop': '20px'})
], style={'padding': '20px'})


# ==========================================
# CALLBACKS DE INTERATIVIDADE
# ==========================================

@app.callback(
    [Output('dim-dropdown-metodo', 'options'),
     Output('dim-dropdown-metodo', 'value'),
     Output('dim-dropdown-epoca', 'options'),
     Output('dim-dropdown-epoca', 'value')],
    [Input('dim-dropdown-dataset', 'value')]
)
def update_controls_by_dataset(dataset_sel):
    df_d6, _ = carregar_dados_projecao(dataset_sel)

    if not df_d6.empty:
        metodos = sorted(list(df_d6['metodo_xai'].dropna().unique()))
        epocas = sorted(list(df_d6['epoca'].dropna().astype(int).unique()))
    else:
        metodos = ['SHAP', 'PFI', 'LIME', 'IG']
        epocas = [1]

    opt_metodos = [{'label': m, 'value': m} for m in metodos]
    val_metodo = metodos[0] if metodos else 'SHAP'

    opt_epocas = [{'label': f"Época {ep}", 'value': ep} for ep in epocas]
    val_epoca = epocas[0] if epocas else 1

    return opt_metodos, val_metodo, opt_epocas, val_epoca


@app.callback(
    [Output('dim-scatter-graph', 'figure'),
     Output('dim-info-instancia', 'children'),
     Output('dim-local-xai-bar', 'figure'),
     Output('dim-tabela-instancias', 'data')],
    [Input('dim-dropdown-dataset', 'value'),
     Input('dim-dropdown-metodo', 'value'),
     Input('dim-dropdown-epoca', 'value'),
     Input('dim-dropdown-algo', 'value'),
     Input('dim-dropdown-cor', 'value'),
     Input('dim-toggle-trajetoria', 'value'),
     Input('dim-scatter-graph', 'clickData'),
     Input('dim-scatter-graph', 'selectedData')]
)
def update_projection(dataset_sel, metodo_sel, epoca_sel, algo_sel, cor_sel, trajetoria_sel, clickData, selectedData):
    df_d6, df_d3 = carregar_dados_projecao(dataset_sel)
    label_ds = MAPA_DATASET.get(dataset_sel, {}).get('label', dataset_sel.upper())

    if df_d6.empty:
        fig_vazia = go.Figure()
        fig_vazia.add_annotation(
            text=f"Aviso: Dados de projeção não encontrados para '{label_ds}'.<br>Execute o gerador para este dataset.",
            showarrow=False, font=dict(size=14, color='red')
        )
        return fig_vazia, "Sem dados.", go.Figure(), []

    # 1. Filtra Dataset 6 para Época e Método
    df_sub = df_d6[
        (df_d6['metodo_xai'] == metodo_sel) &
        (df_d6['epoca'] == epoca_sel)
    ].copy()

    if df_sub.empty:
        fig_vazia = go.Figure()
        fig_vazia.add_annotation(text=f"Sem dados para {metodo_sel} em Época {epoca_sel} ({label_ds})", showarrow=False)
        return fig_vazia, "Sem dados.", go.Figure(), []

    # 2. Recálculo PCA
    if algo_sel == 'PCA' and not df_d3.empty:
        df_exp_ep = df_d3[(df_d3['metodo_xai'] == metodo_sel) & (df_d3['epoca'] == epoca_sel)]
        if not df_exp_ep.empty:
            pivot_pca = df_exp_ep.pivot_table(index='instancia_id', columns='feature', values='valor_atribuicao').fillna(0)
            pca = PCA(n_components=2)
            res_pca = pca.fit_transform(pivot_pca.values)
            pca_df = pd.DataFrame(res_pca, index=pivot_pca.index, columns=['dim_1', 'dim_2']).reset_index()
            df_sub = df_sub.merge(pca_df, on='instancia_id', how='left')
            x_col, y_col = 'dim_1', 'dim_2'
        else:
            x_col, y_col = 'tsne_dim_1', 'tsne_dim_2'
    else:
        x_col, y_col = 'tsne_dim_1', 'tsne_dim_2'

    # 3. Identifica Instância Clicada
    instancia_clicada_id = None
    if clickData and 'points' in clickData and len(clickData['points']) > 0:
        instancia_clicada_id = clickData['points'][0]['customdata'][0]

    # 4. Scatter Plot
    if cor_sel == 'erro_predicao':
        df_sub['cor_label'] = df_sub['erro_predicao'].map({0: 'Acerto', 1: 'Erro'})
        fig = px.scatter(
            df_sub, x=x_col, y=y_col,
            color='cor_label',
            color_discrete_map={'Acerto': '#2ca02c', 'Erro': '#d62728'},
            custom_data=['instancia_id', 'classe_real', 'classe_predita', 'erro_predicao']
        )
    else:
        fig = px.scatter(
            df_sub, x=x_col, y=y_col,
            color=cor_sel,
            color_continuous_scale='Viridis',
            custom_data=['instancia_id', 'classe_real', 'classe_predita', 'erro_predicao']
        )

    fig.update_traces(
        marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color='black')),
        hovertemplate=(
            "<b>Instância ID:</b> #%{customdata[0]}<br>"
            "<b>Classe Real:</b> %{customdata[1]}<br>"
            "<b>Classe Predita:</b> %{customdata[2]}<br>"
            "<b>Erro:</b> %{customdata[3]}<br>"
            "<b>Coord X:</b> %{x:.3f}<br>"
            "<b>Coord Y:</b> %{y:.3f}<extra></extra>"
        )
    )

    # 5. Trajetória Temporal
    if 'ATIVO' in trajetoria_sel and instancia_clicada_id is not None:
        df_traj = df_d6[
            (df_d6['metodo_xai'] == metodo_sel) &
            (df_d6['instancia_id'] == instancia_clicada_id)
        ].sort_values('epoca')

        fig.add_trace(go.Scatter(
            x=df_traj['tsne_dim_1'],
            y=df_traj['tsne_dim_2'],
            mode='lines+markers+text',
            text=[f"E{ep}" for ep in df_traj['epoca']],
            textposition="top center",
            line=dict(color='orange', width=3, dash='dash'),
            marker=dict(size=12, color='orange', symbol='diamond'),
            name=f'Trajetória ID #{instancia_clicada_id}'
        ))

    if instancia_clicada_id is not None:
        ponto_clicado = df_sub[df_sub['instancia_id'] == instancia_clicada_id]
        if not ponto_clicado.empty:
            fig.add_trace(go.Scatter(
                x=ponto_clicado[x_col],
                y=ponto_clicado[y_col],
                mode='markers',
                marker=dict(size=18, color='rgba(0,0,0,0)', line=dict(color='yellow', width=3)),
                showlegend=False,
                hoverinfo='skip'
            ))

    fig.update_layout(
        title=f"Espaço de Explicação 2D ({algo_sel}) - Época {epoca_sel} ({metodo_sel}) - {label_ds}",
        xaxis_title=f"{algo_sel} Dim 1",
        yaxis_title=f"{algo_sel} Dim 2",
        margin=dict(l=20, r=20, t=50, b=20)
    )

    # 6. Painel Lateral Local
    fig_xai_local = go.Figure()
    info_html = "Clique em um ponto no gráfico para inspecionar a instância."

    if instancia_clicada_id is not None:
        row_inst = df_sub[df_sub['instancia_id'] == instancia_clicada_id]
        if not row_inst.empty:
            c_real = row_inst['classe_real'].values[0]
            c_pred = row_inst['classe_predita'].values[0]
            err = row_inst['erro_predicao'].values[0]

            info_html = html.Div([
                html.P([html.B("ID da Instância: "), f"#{instancia_clicada_id}"]),
                html.P([html.B("Classe Real: "), f"{c_real}"]),
                html.P([html.B("Classe Predita: "), f"{c_pred}"]),
                html.P([html.B("Status: "), html.Span("ERRO", style={'color': 'red', 'fontWeight': 'bold'}) if err == 1 else html.Span("ACERTO", style={'color': 'green', 'fontWeight': 'bold'})])
            ])

        if not df_d3.empty:
            df_local_feat = df_d3[
                (df_d3['metodo_xai'] == metodo_sel) &
                (df_d3['epoca'] == epoca_sel) &
                (df_d3['instancia_id'] == instancia_clicada_id)
            ].sort_values('valor_atribuicao', key=abs, ascending=True).tail(10)

            if not df_local_feat.empty:
                colors_bar = ['#d62728' if v < 0 else '#1f77b4' for v in df_local_feat['valor_atribuicao']]
                fig_xai_local.add_trace(go.Bar(
                    x=df_local_feat['valor_atribuicao'],
                    y=df_local_feat['feature'],
                    orientation='h',
                    marker_color=colors_bar
                ))
                fig_xai_local.update_layout(
                    title=f"Top 10 Features ({metodo_sel}) - Ép {epoca_sel}",
                    xaxis_title="Atribuição de Importância",
                    yaxis_title="Feature",
                    margin=dict(l=20, r=20, t=40, b=20)
                )

    # 7. Tabela Auxiliar (Brushing)
    if selectedData and 'points' in selectedData:
        selected_ids = [p['customdata'][0] for p in selectedData['points'] if 'customdata' in p]
        df_tabela = df_sub[df_sub['instancia_id'].isin(selected_ids)]
    else:
        df_tabela = df_sub

    tabela_records = df_tabela.to_dict('records')

    return fig, info_html, fig_xai_local, tabela_records

if __name__ == '__main__':
    app.run(debug=True, port=8052)

<IPython.core.display.Javascript object>

In [14]:

import zipfile
import os
from google.colab import files

# Nome do ficheiro ZIP de destino
zip_filename = "datasets_xai_resultados.zip"

# Cria o ficheiro ZIP contendo todos os CSVs gerados na pasta atual
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in os.listdir('.'):
        if file.endswith('.csv'):
            zipf.write(file)
            print(f"Adicionado ao ZIP: {file}")

# Dispara o descarregamento automático do ZIP para o seu computador
files.download(zip_filename)


Adicionado ao ZIP: metricas_estabilidade_concordancia_german_credit.csv
Adicionado ao ZIP: projecao_tsne_epocas_german_credit.csv
Adicionado ao ZIP: projecao_tsne_epocas_breast_cancer.csv
Adicionado ao ZIP: modelos_german_credit.csv
Adicionado ao ZIP: projecao_tsne_epocas_wine_quality.csv
Adicionado ao ZIP: metricas_epocas_german_credit.csv
Adicionado ao ZIP: metricas_epocas_breast_cancer.csv
Adicionado ao ZIP: resumo_explicacoes_wine_quality.csv
Adicionado ao ZIP: explicacoes_instancias_german_credit.csv
Adicionado ao ZIP: resumo_explicacoes_breast_cancer.csv
Adicionado ao ZIP: explicacoes_instancias_wine_quality.csv
Adicionado ao ZIP: metricas_estabilidade_concordancia_wine_quality.csv
Adicionado ao ZIP: explicacoes_instancias_breast_cancer.csv
Adicionado ao ZIP: modelos_wine_quality.csv
Adicionado ao ZIP: modelos_breast_cancer.csv
Adicionado ao ZIP: resumo_explicacoes_german_credit.csv
Adicionado ao ZIP: metricas_estabilidade_concordancia_breast_cancer.csv
Adicionado ao ZIP: metrica

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>